# Test de Ablación: Label Leakage y Horizon Leakage

Este notebook evalúa la hipótesis de que el modelo BiLSTM con aprendizaje multitarea (predicción de sepsis + estimación de SOFA) puede estar aprendiendo indirectamente la propia definición operacional de sepsis (ΔSOFA ≥ 2) en lugar de predecir el deterioro fisiopatológico genuino.

## Experimentos de ablación

1. **Modelo Base**: Todas las variables (26) + tarea auxiliar SOFA (λ=0.2)
2. **Modelo sin variables SOFA-directas**: Excluye variables directamente relacionadas con el cálculo del SOFA, manteniendo la tarea auxiliar
3. **Modelo sin tarea auxiliar SOFA**: Mantiene todas las variables pero desactiva la cabeza de regresión SOFA (λ=0)

Si el modelo depende fuertemente de las variables SOFA-directas o de la tarea auxiliar para alcanzar su rendimiento, ello indicaría que está detectando el estado fisiológico ya comprometido (label leakage) en lugar de anticipar el deterioro.

In [ ]:
import warnings
import pandas as pd
import numpy as np
import torch
import torch.nn as nn

from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (
    roc_auc_score,
    roc_curve,
    average_precision_score
)

warnings.filterwarnings('ignore')

print(f'PyTorch: {torch.__version__}')
print(f'CUDA disponible: {torch.cuda.is_available()}')

PyTorch: 2.12.0+cu132
CUDA disponible: True


In [ ]:
# Configuración
BASE_DATA_DIR = '../data'
OUTPUT_DIR    = '../data/processed'
MODELS_DIR    = '../models'
RESULTS_DIR   = '../results'

SEQ_LEN = 4
MAX_SEQ_LEN = 24
PREDICTION_HORIZON_H = 6
MIN_HOURS = SEQ_LEN

BATCH_SIZE = 256
EPOCHS = 80 # Reducido para ablación (vs 150 en modelo final)
LR = 1e-3
WEIGHT_DECAY = 1e-4
HIDDEN_SIZE = 64
NUM_LAYERS = 1
DROPOUT = 0.4
PATIENCE = 20 # Reducido para ablación
WARMUP_EPOCHS = 5

RANDOM_STATE_SEED = 42
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

torch.manual_seed(RANDOM_STATE_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_STATE_SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

Path(MODELS_DIR).mkdir(parents=True, exist_ok=True)
Path(RESULTS_DIR).mkdir(parents=True, exist_ok=True)

print(f'Dispositivo: {DEVICE}')
print(f'MAX_SEQ_LEN: {MAX_SEQ_LEN}h | HIDDEN_SIZE: {HIDDEN_SIZE}')

Dispositivo: cuda
MAX_SEQ_LEN: 24h | HIDDEN_SIZE: 64


In [ ]:
# Carga de datos
features_df = pd.read_parquet(f'{OUTPUT_DIR}/features_df.parquet')
cohort_df = pd.read_parquet(f'{OUTPUT_DIR}/cohort_df.parquet')
sofa_hourly = pd.read_parquet(f'{OUTPUT_DIR}/sofa_hourly.parquet')

FEATURE_COLS = [
    col for col in features_df.columns
    if col not in ['stay_id', 'hour_bucket', 'Bands']
]
N_FEATURES = len(FEATURE_COLS)
print(f'Features totales ({N_FEATURES}): {FEATURE_COLS}')

# Variables SOFA-directas a excluir en el experimento de ablación
SOFA_DIRECT_COLS = [
    'GCS - Motor Response',
    'GCS - Verbal Response',
    'Creatinine',
    'Bilirubin, Total',
    'Platelet Count',
    'urine_output_ml',
    'vasopressor_active',
    'mechanical_ventilation',
    'pO2',
    'Arterial Blood Pressure mean',
]
NON_SOFA_COLS = [c for c in FEATURE_COLS if c not in SOFA_DIRECT_COLS]
print(f'\nVariables SOFA-directas ({len(SOFA_DIRECT_COLS)}): {SOFA_DIRECT_COLS}')
print(f'Variables NO-SOFA ({len(NON_SOFA_COLS)}): {NON_SOFA_COLS}')

# Preparación de cohorte
cohort_df['intime'] = pd.to_datetime(cohort_df['intime'])
cohort_df['sepsis_onset'] = pd.to_datetime(cohort_df['sepsis_onset'])
cohort_df['onset_h'] = np.where(
    cohort_df['label'] == 1,
    (cohort_df['sepsis_onset'] - cohort_df['intime']).dt.total_seconds() / 3600,
    np.nan
)
ONSET_H_MAP = cohort_df.set_index('stay_id')['onset_h'].to_dict()
LABEL_MAP   = cohort_df.set_index('stay_id')['label'].to_dict()

sofa_hourly['hour_bucket'] = sofa_hourly['h_since_intime'].astype(int)
SOFA_LOOKUP = sofa_hourly.set_index(['stay_id', 'hour_bucket'])['sofa'].to_dict()

print(f'\nEstancias totales: {cohort_df["stay_id"].nunique():,}')
print(f'  Sepsis: {(cohort_df["label"] == 1).sum():,}')
print(f'  Control: {(cohort_df["label"] == 0).sum():,}')

Features totales (26): ['Arterial Blood Pressure mean', 'GCS - Motor Response', 'GCS - Verbal Response', 'Heart Rate', 'Non Invasive Blood Pressure diastolic', 'Non Invasive Blood Pressure systolic', 'O2 saturation pulseoxymetry', 'PEEP set', 'Respiratory Rate', 'Temperature Celsius', 'Bicarbonate', 'Bilirubin, Total', 'Creatinine', 'Glucose', 'Lactate', 'Platelet Count', 'Urea Nitrogen', 'White Blood Cells', 'pH', 'pO2', 'urine_output_ml', 'vasopressor_active', 'mechanical_ventilation', 'age', 'gender_enc', 'charlson_score']

Variables SOFA-directas (10): ['GCS - Motor Response', 'GCS - Verbal Response', 'Creatinine', 'Bilirubin, Total', 'Platelet Count', 'urine_output_ml', 'vasopressor_active', 'mechanical_ventilation', 'pO2', 'Arterial Blood Pressure mean']
Variables NO-SOFA (16): ['Heart Rate', 'Non Invasive Blood Pressure diastolic', 'Non Invasive Blood Pressure systolic', 'O2 saturation pulseoxymetry', 'PEEP set', 'Respiratory Rate', 'Temperature Celsius', 'Bicarbonate', 'Glucose

In [ ]:
# Split train/val/test (idéntico al modelo base)
stays_coverage = features_df.groupby('stay_id')['hour_bucket'].count()
valid_stay_ids = stays_coverage[stays_coverage >= MIN_HOURS].index

stay_labels_v = cohort_df[cohort_df['stay_id'].isin(valid_stay_ids)][
    ['stay_id', 'label', 'subject_id']
].reset_index(drop=True)

all_stays    = stay_labels_v['stay_id'].values
all_labels   = stay_labels_v['label'].values
all_subjects = stay_labels_v['subject_id'].values

first_split = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE_SEED)
tv_idx, test_idx = next(first_split.split(all_stays, all_labels, groups=all_subjects))

second_split = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=RANDOM_STATE_SEED)
tr_idx, val_idx = next(second_split.split(all_stays[tv_idx], all_labels[tv_idx], groups=all_subjects[tv_idx]))

train_stays = all_stays[tv_idx][tr_idx]
val_stays   = all_stays[tv_idx][val_idx]
test_stays  = all_stays[test_idx]

print(f'Train: {len(train_stays):,} estancias')
print(f'Val:   {len(val_stays):,} estancias')
print(f'Test:  {len(test_stays):,} estancias')

Train: 44,794 estancias
Val:   14,889 estancias
Test:  14,871 estancias


In [ ]:
# Función de ventaneo rodante
def build_sequences(features_df, stay_ids, feature_cols, max_seq_len=24, horizon=6):
    rows = []
    for sid in stay_ids:
        df_s = features_df[features_df['stay_id'] == sid].sort_values('hour_bucket')
        vals = df_s[feature_cols].values
        hours = df_s['hour_bucket'].values
        n = len(vals)
        for i in range(n):
            start = max(0, i - max_seq_len + 1)
            seq = vals[start:i+1]
            t = hours[i]
            # Padding izquierdo
            if len(seq) < max_seq_len:
                pad = np.zeros((max_seq_len - len(seq), len(feature_cols)), dtype=np.float32)
                seq = np.concatenate([pad, seq], axis=0)
            real_len = min(max_seq_len, i + 1)
            onset = ONSET_H_MAP.get(sid, np.nan)
            y = 0 if pd.isna(onset) else int((t + horizon) >= onset and t < onset)
            sofa = SOFA_LOOKUP.get((sid, int(t)), np.nan)
            lead = np.nan if pd.isna(onset) else (onset - t)
            rows.append((seq.astype(np.float32), y, real_len, sofa, lead, sid))
    return rows

# Preparar datos con TODAS las variables
print('Construyendo secuencias con TODAS las variables...')
train_rows_all = build_sequences(features_df, train_stays, FEATURE_COLS)
val_rows_all   = build_sequences(features_df, val_stays,   FEATURE_COLS)
test_rows_all  = build_sequences(features_df, test_stays,  FEATURE_COLS)

# Preparar datos SIN variables SOFA-directas
print('Construyendo secuencias SIN variables SOFA-directas...')
train_rows_nosofa = build_sequences(features_df, train_stays, NON_SOFA_COLS)
val_rows_nosofa   = build_sequences(features_df, val_stays,   NON_SOFA_COLS)
test_rows_nosofa  = build_sequences(features_df, test_stays,  NON_SOFA_COLS)

def rows_to_arrays(rows):
    X = np.stack([r[0] for r in rows])
    y = np.array([r[1] for r in rows], dtype=np.float32)
    lens = np.array([r[2] for r in rows], dtype=np.int32)
    sofa = np.array([r[3] for r in rows], dtype=np.float32)
    leads = np.array([r[4] for r in rows], dtype=np.float32)
    return X, y, lens, sofa, leads

X_train_all, y_train_all, lens_train_all, sofa_train_all, leads_train_all = rows_to_arrays(train_rows_all)
X_val_all,   y_val_all,   lens_val_all,   sofa_val_all,   leads_val_all   = rows_to_arrays(val_rows_all)
X_test_all,  y_test_all,  lens_test_all,  sofa_test_all,  leads_test_all  = rows_to_arrays(test_rows_all)

X_train_ns, y_train_ns, lens_train_ns, sofa_train_ns, leads_train_ns = rows_to_arrays(train_rows_nosofa)
X_val_ns,   y_val_ns,   lens_val_ns,   sofa_val_ns,   leads_val_ns   = rows_to_arrays(val_rows_nosofa)
X_test_ns,  y_test_ns,  lens_test_ns,  sofa_test_ns,  leads_test_ns  = rows_to_arrays(test_rows_nosofa)

print(f'\nTODAS vars — Train: {X_train_all.shape} | Val: {X_val_all.shape} | Test: {X_test_all.shape}')
print(f'SIN SOFA  — Train: {X_train_ns.shape}  | Val: {X_val_ns.shape}  | Test: {X_test_ns.shape}')

Construyendo secuencias con TODAS las variables...
Construyendo secuencias SIN variables SOFA-directas...

TODAS vars — Train: (4107587, 24, 26) | Val: (1360982, 24, 26) | Test: (1352642, 24, 26)
SIN SOFA  — Train: (4107587, 24, 16)  | Val: (1360982, 24, 16)  | Test: (1352642, 24, 16)


In [ ]:
# Subsampling estratificado de negativos (ratio 10:1)
NEG_POS_RATIO = 10

def subsample_negatives(X, y, sids, lens, leads, sofa):
    pos_idx = np.where(y == 1)[0]
    neg_idx = np.where(y == 0)[0]
    n_neg_keep = min(len(neg_idx), len(pos_idx) * NEG_POS_RATIO)
    _rng = np.random.default_rng(RANDOM_STATE_SEED)
    neg_keep = _rng.choice(neg_idx, size=n_neg_keep, replace=False)
    keep = np.sort(np.concatenate([pos_idx, neg_keep]))
    return X[keep], y[keep], sids[keep], lens[keep], leads[keep], sofa[keep]

# --- TRAIN subsample (sobre datos crudos) ---
sids_train_all = np.array([r[5] for r in train_rows_all], dtype=np.int32)
X_train_all_sub, y_train_all_sub, sids_train_all_sub, lens_train_all_sub, leads_train_all_sub, sofa_train_all_sub = subsample_negatives(
    X_train_all, y_train_all, sids_train_all, lens_train_all, leads_train_all, sofa_train_all
)

sids_train_ns = np.array([r[5] for r in train_rows_nosofa], dtype=np.int32)
X_train_ns_sub, y_train_ns_sub, sids_train_ns_sub, lens_train_ns_sub, leads_train_ns_sub, sofa_train_ns_sub = subsample_negatives(
    X_train_ns, y_train_ns, sids_train_ns, lens_train_ns, leads_train_ns, sofa_train_ns
)

print(f'Train ALL subsampled -- X: {X_train_all_sub.shape} | pos: {y_train_all_sub.mean():.2%}')
print(f'Train NO-SOFA subsampled -- X: {X_train_ns_sub.shape} | pos: {y_train_ns_sub.mean():.2%}')
# Preprocesamiento: forward fill, deltas, missingness mask, normalización
def apply_forward_fill_vectorized(X, max_gap=24):
    n, T, F = X.shape
    X_ff = X.copy()
    arange_T = np.arange(T)
    for j in range(F):
        arr = X_ff[:, :, j]
        mask = ~np.isnan(arr)
        idx = np.where(mask, arange_T, 0)
        np.maximum.accumulate(idx, axis=1, out=idx)
        gap = arange_T[None, :] - idx
        filled = arr[np.arange(n)[:, None], idx]
        filled = np.where(gap > max_gap, np.nan, filled)
        X_ff[:, :, j] = filled
    return X_ff

def compute_deltas(X_ff, M_original):
    deltas = np.zeros_like(X_ff)
    diff = X_ff[:, 1:, :] - X_ff[:, :-1, :]
    deltas[:, 1:, :] = np.where(np.isnan(diff), 0.0, diff)
    both = M_original[:, 1:, :] * M_original[:, :-1, :]
    deltas[:, 1:, :] *= both
    return deltas.astype(np.float32)

def apply_missingness_mask(X, train_medians=None, external_mask=None, external_deltas=None):
    _, _, F = X.shape
    M = external_mask if external_mask is not None else (~np.isnan(X)).astype(np.float32)
    X_imp = X.copy()
    if train_medians is None:
        flat = X_imp.reshape(-1, F)
        train_medians = np.nanmedian(flat, axis=0)
        train_medians = np.where(np.isnan(train_medians), 0.0, train_medians)
    for j in range(F):
        mask_nan = np.isnan(X_imp[:, :, j])
        X_imp[:, :, j][mask_nan] = train_medians[j]
    parts = [X_imp]
    if external_deltas is not None:
        parts.append(external_deltas)
    parts.append(M)
    return np.concatenate(parts, axis=2).astype(np.float32), train_medians

def scale_masked(X, scaler, F, fit=False):
    n, T, F_total = X.shape
    has_deltas = (F_total == 3 * F)
    X_vals = X[:, :, :F].reshape(-1, F)
    if fit:
        X_vals_scaled = scaler.fit_transform(X_vals)
    else:
        X_vals_scaled = scaler.transform(X_vals)
    X_vals_scaled = X_vals_scaled.reshape(n, T, F).astype(np.float32)
    if has_deltas:
        delta_std = np.where(scaler.scale_ > 0, scaler.scale_, 1.0)
        X_deltas = X[:, :, F:2*F].reshape(-1, F)
        X_deltas_scaled = (X_deltas / delta_std).reshape(n, T, F).astype(np.float32)
        X_mask = X[:, :, 2*F:]
        return np.concatenate([X_vals_scaled, X_deltas_scaled, X_mask], axis=2)
    else:
        X_mask = X[:, :, F:]
        return np.concatenate([X_vals_scaled, X_mask], axis=2)

def preprocess_pipeline_chunked(X_train, X_val, X_test, chunk_size=200000):
    # Train: preprocesar completo (subsampleado, cabe en memoria)
    M_train = (~np.isnan(X_train)).astype(np.float32)
    X_train_ff = apply_forward_fill_vectorized(X_train)
    deltas_train = compute_deltas(X_train_ff, M_train)
    X_train_masked, train_medians = apply_missingness_mask(X_train_ff, external_mask=M_train, external_deltas=deltas_train)
    F = X_train.shape[2]
    scaler = StandardScaler()
    X_train_norm = scale_masked(X_train_masked, scaler, F, fit=True)
    X_train_norm = np.nan_to_num(X_train_norm, nan=0.0, posinf=0.0, neginf=0.0)
    del M_train, X_train_ff, deltas_train, X_train_masked
    import gc; gc.collect()

    def preprocess_split(X_split):
        n = len(X_split)
        result_chunks = []
        for i in range(0, n, chunk_size):
            X_chunk = X_split[i:i+chunk_size]
            M_chunk = (~np.isnan(X_chunk)).astype(np.float32)
            X_chunk_ff = apply_forward_fill_vectorized(X_chunk)
            deltas_chunk = compute_deltas(X_chunk_ff, M_chunk)
            X_chunk_masked, _ = apply_missingness_mask(X_chunk_ff, train_medians, external_mask=M_chunk, external_deltas=deltas_chunk)
            X_chunk_norm = scale_masked(X_chunk_masked, scaler, F, fit=False)
            X_chunk_norm = np.nan_to_num(X_chunk_norm, nan=0.0, posinf=0.0, neginf=0.0)
            result_chunks.append(X_chunk_norm)
            del X_chunk, M_chunk, X_chunk_ff, deltas_chunk, X_chunk_masked
            gc.collect()
        return np.concatenate(result_chunks, axis=0)

    X_val_norm = preprocess_split(X_val)
    X_test_norm = preprocess_split(X_test)
    return X_train_norm, X_val_norm, X_test_norm, F

print('Preprocesando TODAS las variables (train subsampleado, val/test completos por chunks)...')
X_tr_a, X_va_a, X_te_a, F_all = preprocess_pipeline_chunked(X_train_all_sub, X_val_all, X_test_all)
print(f'  -> F={F_all}, train={X_tr_a.shape}, val={X_va_a.shape}, test={X_te_a.shape}')

print('Preprocesando SIN variables SOFA-directas...')
X_tr_ns, X_va_ns, X_te_ns, F_ns = preprocess_pipeline_chunked(X_train_ns_sub, X_val_ns, X_test_ns)
print(f'  -> F={F_ns}, train={X_tr_ns.shape}, val={X_va_ns.shape}, test={X_te_ns.shape}')


Train ALL subsampled -- X: (692461, 24, 26) | pos: 9.09%
Train NO-SOFA subsampled -- X: (692461, 24, 16) | pos: 9.09%
Preprocesando TODAS las variables (train subsampleado, val/test completos por chunks)...
  -> F=26, train=(692461, 24, 78), val=(1360982, 24, 78), test=(1352642, 24, 78)
Preprocesando SIN variables SOFA-directas...
  -> F=16, train=(692461, 24, 48), val=(1360982, 24, 48), test=(1352642, 24, 48)


In [ ]:
# Datasets y DataLoaders
class SepsisDataset(Dataset):
    def __init__(self, X, y, lengths, sofa):
        self.X = torch.from_numpy(X).float()
        self.y = torch.from_numpy(y).float()
        self.lengths = torch.from_numpy(lengths).long()
        self.sofa = torch.from_numpy(sofa).float()
    def __len__(self):
        return len(self.y)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx], self.lengths[idx], self.sofa[idx]

# ALL vars loaders
train_ds_all = SepsisDataset(X_tr_a, y_train_all_sub, lens_train_all_sub, sofa_train_all_sub)
val_ds_all   = SepsisDataset(X_va_a, y_val_all, lens_val_all, sofa_val_all)
test_ds_all  = SepsisDataset(X_te_a, y_test_all, lens_test_all, sofa_test_all)
train_loader_all = DataLoader(train_ds_all, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader_all   = DataLoader(val_ds_all,   batch_size=BATCH_SIZE, shuffle=False)
test_loader_all  = DataLoader(test_ds_all,  batch_size=BATCH_SIZE, shuffle=False)

# NO-SOFA vars loaders
train_ds_ns = SepsisDataset(X_tr_ns, y_train_ns_sub, lens_train_ns_sub, sofa_train_ns_sub)
val_ds_ns   = SepsisDataset(X_va_ns, y_val_ns, lens_val_ns, sofa_val_ns)
test_ds_ns  = SepsisDataset(X_te_ns, y_test_ns, lens_test_ns, sofa_test_ns)
train_loader_ns = DataLoader(train_ds_ns, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader_ns   = DataLoader(val_ds_ns,   batch_size=BATCH_SIZE, shuffle=False)
test_loader_ns  = DataLoader(test_ds_ns,  batch_size=BATCH_SIZE, shuffle=False)

print(f'ALL   -- train batches: {len(train_loader_all)} | val batches: {len(val_loader_all)} | test batches: {len(test_loader_all)}')
print(f'NO-SOFA -- train batches: {len(train_loader_ns)}  | val batches: {len(val_loader_ns)}  | test batches: {len(test_loader_ns)}')

# Verificar distribución de clases en val
print(f'Val ALL class distribution: {np.bincount(y_val_all.astype(int))}')
print(f'Val NO-SOFA class distribution: {np.bincount(y_val_ns.astype(int))}')


ALL   -- train batches: 2704 | val batches: 5317 | test batches: 5284
NO-SOFA -- train batches: 2704  | val batches: 5317  | test batches: 5284
Val ALL class distribution: [1339719   21263]
Val NO-SOFA class distribution: [1339719   21263]


In [ ]:
# Definición del modelo (igual que el original, con lambda configurable)
class Bi_LSTM_Sepsis(nn.Module):
    def __init__(self, input_size, hidden_size=HIDDEN_SIZE, num_layers=NUM_LAYERS, dropout=DROPOUT):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size, hidden_size=hidden_size, num_layers=num_layers,
            batch_first=True, dropout=dropout if num_layers > 1 else 0.0, bidirectional=True
        )
        lstm_out = hidden_size * 2
        self.attn    = nn.Linear(lstm_out, 1)
        self.bn1     = nn.BatchNorm1d(lstm_out)
        self.dropout = nn.Dropout(dropout)
        self.fc1     = nn.Linear(lstm_out, 64)
        self.relu    = nn.ReLU()
        self.fc2     = nn.Linear(64, 1)
        self.fc_sofa = nn.Linear(64, 1)

    def forward(self, x, lengths=None):
        out, _ = self.lstm(x)
        attn_logits = self.attn(out).squeeze(-1)
        if lengths is not None:
            T = out.size(1)
            positions = torch.arange(T, device=x.device).unsqueeze(0)
            pad_start = T - lengths.unsqueeze(1)
            real_mask = positions >= pad_start
            attn_logits = attn_logits.masked_fill(~real_mask, float('-inf'))
        attn_weights = torch.softmax(attn_logits, dim=1).unsqueeze(-1)
        context = (attn_weights * out).sum(dim=1)
        context = self.bn1(context)
        context = self.dropout(context)
        shared = self.relu(self.fc1(context))
        sepsis_logit = self.fc2(shared).squeeze(-1)
        sofa_pred    = self.fc_sofa(shared).squeeze(-1)
        return sepsis_logit, sofa_pred

class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
    def forward(self, logits, targets):
        bce = nn.functional.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        p = torch.sigmoid(logits)
        p_t = p * targets + (1 - p) * (1 - targets)
        alpha_t = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        focal = alpha_t * (1 - p_t) ** self.gamma * bce
        return focal.mean()

def sofa_auxiliary_loss(sofa_pred, sofa_true):
    mask = ~torch.isnan(sofa_true)
    if not mask.any():
        return torch.tensor(0.0, device=sofa_pred.device)
    return nn.functional.mse_loss(sofa_pred[mask], sofa_true[mask] / 24.0)

print('Modelo y funciones de pérdida definidos.')

Modelo y funciones de pérdida definidos.


In [ ]:
# Función de entrenamiento con lambda configurable
def train_model_ablation(model, train_loader, val_loader, model_name, lambda_sofa=0.2):
    focal_loss = FocalLoss(alpha=0.25, gamma=2.0)
    optimizer  = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler  = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', patience=7, factor=0.5)

    history = {'train_loss': [], 'val_loss': [], 'val_auroc': []}
    best_val_auroc = 0.0
    patience_counter = 0
    best_path = f'{MODELS_DIR}/{model_name}.pt'

    for epoch in range(1, EPOCHS + 1):
        if epoch <= WARMUP_EPOCHS:
            warmup_lr = LR * epoch / WARMUP_EPOCHS
            for g in optimizer.param_groups:
                g['lr'] = warmup_lr

        model.train()
        train_loss = 0.0
        train_samples = 0
        for X_b, y_b, len_b, sofa_b in train_loader:
            X_b, y_b, len_b, sofa_b = X_b.to(DEVICE), y_b.to(DEVICE), len_b.to(DEVICE), sofa_b.to(DEVICE)
            optimizer.zero_grad()
            sepsis_logit, sofa_pred = model(X_b, len_b)
            loss = focal_loss(sepsis_logit, y_b) + lambda_sofa * sofa_auxiliary_loss(sofa_pred, sofa_b)
            if torch.isnan(loss):
                print(f'WARNING: NaN loss en train epoch {epoch}')
                continue
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_loss += loss.item() * len(y_b)
            train_samples += len(y_b)
        train_loss /= max(train_samples, 1)

        model.eval()
        val_loss = 0.0
        val_probs, val_true = [], []
        with torch.no_grad():
            for X_b, y_b, len_b, sofa_b in val_loader:
                X_b, y_b, len_b, sofa_b = X_b.to(DEVICE), y_b.to(DEVICE), len_b.to(DEVICE), sofa_b.to(DEVICE)
                sepsis_logit, sofa_pred = model(X_b, len_b)
                loss = focal_loss(sepsis_logit, y_b) + lambda_sofa * sofa_auxiliary_loss(sofa_pred, sofa_b)
                val_loss += loss.item() * len(y_b)
                probs = torch.sigmoid(sepsis_logit).cpu().numpy()
                val_probs.extend(probs)
                val_true.extend(y_b.cpu().numpy())
        val_loss /= len(val_loader.dataset)

        val_probs_arr = np.array(val_probs, dtype=np.float64)
        val_true_arr  = np.array(val_true, dtype=np.int32)
        valid_mask = ~(np.isnan(val_probs_arr) | np.isnan(val_true_arr))

        # Diagnostico en primera epoca
        if epoch == 1:
            print(f'  [DIAG] valid_mask.sum()={valid_mask.sum()}, total={len(valid_mask)}')
            print(f'  [DIAG] val_true unique: {np.unique(val_true_arr[valid_mask])}')
            print(f'  [DIAG] val_probs range: [{val_probs_arr[valid_mask].min():.4f}, {val_probs_arr[valid_mask].max():.4f}]')

        if valid_mask.sum() > 0 and len(set(val_true_arr[valid_mask])) > 1:
            val_auroc = roc_auc_score(val_true_arr[valid_mask], val_probs_arr[valid_mask])
        else:
            val_auroc = 0.0
            if epoch == 1:
                print(f'  [DIAG] AUROC forzado a 0.0: valid_mask={valid_mask.sum()}, clases={len(set(val_true_arr[valid_mask]))}')

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_auroc'].append(val_auroc)

        old_lr = optimizer.param_groups[0]['lr']
        if epoch > WARMUP_EPOCHS:
            scheduler.step(val_auroc)
        new_lr = optimizer.param_groups[0]['lr']

        if val_auroc > best_val_auroc:
            best_val_auroc = val_auroc
            patience_counter = 0
            torch.save(model.state_dict(), best_path)
        else:
            patience_counter += 1

        if epoch % 5 == 0 or patience_counter == 0:
            print(f'[{model_name:25s}] Epoch {epoch:3d} | Train: {train_loss:.4f} | Val: {val_loss:.4f} | AUROC: {val_auroc:.4f} | LR: {new_lr:.2e} | P: {patience_counter}')

        if patience_counter >= PATIENCE:
            print(f'[{model_name:25s}] Early stopping en epoch {epoch}.')
            break

    print(f'[{model_name:25s}] Mejor val_auroc: {best_val_auroc:.4f}')
    return {'history': history, 'path': best_path, 'model': model}

print('Función de entrenamiento lista.')


Función de entrenamiento lista.


## Experimento 1: Modelo Base (TODAS las variables + λ=0.2)

In [10]:
model_base = Bi_LSTM_Sepsis(input_size=F_all * 3).to(DEVICE)
results_base = train_model_ablation(
    model_base, train_loader_all, val_loader_all,
    model_name='ablation_base_all_vars_lambda02',
    lambda_sofa=0.2
)

  [DIAG] valid_mask.sum()=1360982, total=1360982
  [DIAG] val_true unique: [0 1]
  [DIAG] val_probs range: [0.0298, 0.6248]
[ablation_base_all_vars_lambda02] Epoch   1 | Train: 0.0234 | Val: 0.0100 | AUROC: 0.8998 | LR: 2.00e-04 | P: 0
[ablation_base_all_vars_lambda02] Epoch   2 | Train: 0.0197 | Val: 0.0099 | AUROC: 0.9042 | LR: 4.00e-04 | P: 0
[ablation_base_all_vars_lambda02] Epoch   5 | Train: 0.0194 | Val: 0.0090 | AUROC: 0.9009 | LR: 1.00e-03 | P: 3
[ablation_base_all_vars_lambda02] Epoch   7 | Train: 0.0193 | Val: 0.0093 | AUROC: 0.9051 | LR: 1.00e-03 | P: 0
[ablation_base_all_vars_lambda02] Epoch   9 | Train: 0.0193 | Val: 0.0092 | AUROC: 0.9055 | LR: 1.00e-03 | P: 0
[ablation_base_all_vars_lambda02] Epoch  10 | Train: 0.0193 | Val: 0.0094 | AUROC: 0.9043 | LR: 1.00e-03 | P: 1
[ablation_base_all_vars_lambda02] Epoch  13 | Train: 0.0192 | Val: 0.0088 | AUROC: 0.9056 | LR: 1.00e-03 | P: 0
[ablation_base_all_vars_lambda02] Epoch  14 | Train: 0.0192 | Val: 0.0090 | AUROC: 0.9061 | 

## Experimento 2: Modelo SIN variables SOFA-directas (λ=0.2)

In [11]:
model_nosofa = Bi_LSTM_Sepsis(input_size=F_ns * 3).to(DEVICE)
results_nosofa = train_model_ablation(
    model_nosofa, train_loader_ns, val_loader_ns,
    model_name='ablation_no_sofa_vars_lambda02',
    lambda_sofa=0.2
)

  [DIAG] valid_mask.sum()=1360982, total=1360982
  [DIAG] val_true unique: [0 1]
  [DIAG] val_probs range: [0.0331, 0.5748]
[ablation_no_sofa_vars_lambda02] Epoch   1 | Train: 0.0270 | Val: 0.0113 | AUROC: 0.8806 | LR: 2.00e-04 | P: 0
[ablation_no_sofa_vars_lambda02] Epoch   2 | Train: 0.0227 | Val: 0.0114 | AUROC: 0.8830 | LR: 4.00e-04 | P: 0
[ablation_no_sofa_vars_lambda02] Epoch   4 | Train: 0.0223 | Val: 0.0114 | AUROC: 0.8845 | LR: 8.00e-04 | P: 0
[ablation_no_sofa_vars_lambda02] Epoch   5 | Train: 0.0223 | Val: 0.0122 | AUROC: 0.8823 | LR: 1.00e-03 | P: 1
[ablation_no_sofa_vars_lambda02] Epoch  10 | Train: 0.0223 | Val: 0.0104 | AUROC: 0.8846 | LR: 1.00e-03 | P: 0
[ablation_no_sofa_vars_lambda02] Epoch  15 | Train: 0.0223 | Val: 0.0117 | AUROC: 0.8843 | LR: 1.00e-03 | P: 5
[ablation_no_sofa_vars_lambda02] Epoch  19 | Train: 0.0221 | Val: 0.0114 | AUROC: 0.8850 | LR: 5.00e-04 | P: 0
[ablation_no_sofa_vars_lambda02] Epoch  20 | Train: 0.0221 | Val: 0.0111 | AUROC: 0.8870 | LR: 5.00

## Experimento 3: Modelo SIN tarea auxiliar SOFA (TODAS las variables + λ=0)

In [12]:
model_nolambda = Bi_LSTM_Sepsis(input_size=F_all * 3).to(DEVICE)
results_nolambda = train_model_ablation(
    model_nolambda, train_loader_all, val_loader_all,
    model_name='ablation_all_vars_lambda00',
    lambda_sofa=0.0
)

  [DIAG] valid_mask.sum()=1360982, total=1360982
  [DIAG] val_true unique: [0 1]
  [DIAG] val_probs range: [0.0150, 0.6732]
[ablation_all_vars_lambda00] Epoch   1 | Train: 0.0216 | Val: 0.0087 | AUROC: 0.8958 | LR: 2.00e-04 | P: 0
[ablation_all_vars_lambda00] Epoch   2 | Train: 0.0189 | Val: 0.0085 | AUROC: 0.9018 | LR: 4.00e-04 | P: 0
[ablation_all_vars_lambda00] Epoch   5 | Train: 0.0186 | Val: 0.0075 | AUROC: 0.9010 | LR: 1.00e-03 | P: 3
[ablation_all_vars_lambda00] Epoch   6 | Train: 0.0186 | Val: 0.0101 | AUROC: 0.9019 | LR: 1.00e-03 | P: 0
[ablation_all_vars_lambda00] Epoch   9 | Train: 0.0185 | Val: 0.0090 | AUROC: 0.9024 | LR: 1.00e-03 | P: 0
[ablation_all_vars_lambda00] Epoch  10 | Train: 0.0185 | Val: 0.0077 | AUROC: 0.9014 | LR: 1.00e-03 | P: 1
[ablation_all_vars_lambda00] Epoch  14 | Train: 0.0185 | Val: 0.0089 | AUROC: 0.9027 | LR: 1.00e-03 | P: 0
[ablation_all_vars_lambda00] Epoch  15 | Train: 0.0185 | Val: 0.0095 | AUROC: 0.9017 | LR: 1.00e-03 | P: 1
[ablation_all_vars_l

## Evaluación y comparación en conjunto de test

In [13]:
def evaluate_model(model, test_loader, model_path, model_name):
    model.load_state_dict(torch.load(model_path, map_location=DEVICE, weights_only=True))
    model.eval()

    test_probs, test_true = [], []
    with torch.no_grad():
        for X_b, y_b, len_b, _ in test_loader:
            sepsis_logit, _ = model(X_b.to(DEVICE), len_b.to(DEVICE))
            test_probs.extend(torch.sigmoid(sepsis_logit).cpu().numpy())
            test_true.extend(y_b.numpy())

    test_probs = np.array(test_probs)
    test_true  = np.array(test_true)

    fpr, tpr, thresholds = roc_curve(test_true, test_probs)
    best_thresh = thresholds[np.argmax(tpr - fpr)]
    preds = (test_probs >= best_thresh).astype(int)

    auroc = roc_auc_score(test_true, test_probs)
    auprc = average_precision_score(test_true, test_probs)
    acc   = (preds == test_true).mean()
    precision = ((preds == 1) & (test_true == 1)).sum() / max(preds.sum(), 1)
    recall    = ((preds == 1) & (test_true == 1)).sum() / max((test_true == 1).sum(), 1)

    print(f'\n=== {model_name} ===')
    print(f'AUROC:      {auroc:.4f}')
    print(f'AUPRC:      {auprc:.4f}')
    print(f'Accuracy:   {acc:.4f}')
    print(f'Precision:  {precision:.4f}')
    print(f'Recall:     {recall:.4f}')
    print(f'Umbral:     {best_thresh:.3f}')

    return {
        'model_name': model_name,
        'auroc': auroc, 'auprc': auprc, 'acc': acc,
        'precision': precision, 'recall': recall,
        'threshold': best_thresh
    }

print('Función de evaluación definida.')

Función de evaluación definida.


In [ ]:
results_summary = []

# Exp 1: Base
r1 = evaluate_model(results_base['model'], test_loader_all, results_base['path'],
                    'Base (all vars + λ=0.2)')
results_summary.append(r1)

# Exp 2: Sin SOFA vars
r2 = evaluate_model(results_nosofa['model'], test_loader_ns, results_nosofa['path'],
                    'Sin SOFA vars (λ=0.2)')
results_summary.append(r2)

# Exp 3: Sin lambda SOFA
r3 = evaluate_model(results_nolambda['model'], test_loader_all, results_nolambda['path'],
                    'Sin λ SOFA (all vars + λ=0.0)')
results_summary.append(r3)

# Tabla resumen
df_summary = pd.DataFrame(results_summary)
print('\n' + '='*70)
print(df_summary.to_string(index=False))
print('='*70)

# Diferencias respecto al base
base_auroc = df_summary.iloc[0]['auroc']
base_auprc = df_summary.iloc[0]['auprc']
df_summary['ΔAUROC'] = df_summary['auroc'] - base_auroc
df_summary['ΔAUPRC'] = df_summary['auprc'] - base_auprc
print('\nDiferencias respecto al modelo base:')
print(df_summary[['model_name', 'ΔAUROC', 'ΔAUPRC']].to_string(index=False))


=== Base (all vars + λ=0.2) ===
AUROC:      0.9122
AUPRC:      0.2622
Accuracy:   0.8426
Precision:  0.0759
Recall:     0.8322
Umbral:     0.208

=== Sin SOFA vars (λ=0.2) ===
AUROC:      0.8909
AUPRC:      0.2257
Accuracy:   0.8474
Precision:  0.0757
Recall:     0.8016
Umbral:     0.216

=== Sin λ SOFA (all vars + λ=0.0) ===
AUROC:      0.9096
AUPRC:      0.2586
Accuracy:   0.8634
Precision:  0.0843
Recall:     0.8053
Umbral:     0.228

                   model_name    auroc    auprc      acc  precision   recall  threshold
      Base (all vars + λ=0.2) 0.912238 0.262164 0.842608   0.075895 0.832245   0.208142
        Sin SOFA vars (λ=0.2) 0.890865 0.225733 0.847390   0.075681 0.801568   0.216263
Sin λ SOFA (all vars + λ=0.0) 0.909584 0.258563 0.863359   0.084296 0.805342   0.228134

Diferencias respecto al modelo base:
                   model_name    ΔAUROC    ΔAUPRC
      Base (all vars + λ=0.2)  0.000000  0.000000
        Sin SOFA vars (λ=0.2) -0.021372 -0.036431
Sin λ SOFA (all v

## Interpretación de resultados

### Hipótesis de label leakage

Si el modelo base alcanza un AUROC significativamente superior al modelo **sin variables SOFA-directas**, ello indica que el modelo depende fuertemente de información ya contenida en la definición operacional de sepsis (ΔSOFA ≥ 2).

| Escenario | Interpretación |
|-----------|---------------|
| ΔAUROC < -0.05 (caída drástica) | **Label leakage confirmado**: el modelo aprendía la definición SOFA, no anticipaba el deterioro |
| ΔAUROC ≈ -0.01 a -0.03 (caída moderada) | **Dependencia parcial**: las variables SOFA aportan pero no dominan |
| ΔAUROC ≈ 0 (sin cambio) | **No hay label leakage**: el modelo generaliza sin variables SOFA-directas |

### Hipótesis de horizon leakage

Si el modelo base alcanza un AUROC significativamente superior al modelo **sin tarea auxiliar SOFA (λ=0)**, ello sugiere que la cabeza auxiliar actúa como conducto de información del label hacia la predicción.

| Escenario | Interpretación |
|-----------|---------------|
| ΔAUROC < -0.03 con λ=0 | **Horizon leakage parcial**: la multitarea facilita el acceso al estado de disfunción orgánica |
| ΔAUROC ≈ 0 con λ=0 | **La multitarea es benéfica pero no determinante**: el modelo no depende de la cabeza SOFA |


In [ ]:
# Guardar resultados
df_summary.to_csv(f'{RESULTS_DIR}/ablation_test_summary.csv', index=False)
print(f'Resultados guardados en: {RESULTS_DIR}/ablation_test_summary.csv')

# Guardar también como JSON para referencia
import json
with open(f'{RESULTS_DIR}/ablation_test_summary.json', 'w') as f:
    json.dump(results_summary, f, indent=2)
print(f'Resultados JSON guardados en: {RESULTS_DIR}/ablation_test_summary.json')

Resultados guardados en: ../results/ablation_test_summary.csv
Resultados JSON guardados en: ../results/ablation_test_summary.json
